In [1]:
import numpy as np
import pandas as pd

In [2]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [3]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [4]:
# merging both the files
movies = movies.merge(credits, on = "title")

In [5]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [6]:
# Keeping the valid features
movies = movies[['genres', 'movie_id', 'keywords', 'overview', 'title', 'cast', 'crew']]

In [7]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   genres    4809 non-null   object
 1   movie_id  4809 non-null   int64 
 2   keywords  4809 non-null   object
 3   overview  4806 non-null   object
 4   title     4809 non-null   object
 5   cast      4809 non-null   object
 6   crew      4809 non-null   object
dtypes: int64(1), object(6)
memory usage: 263.1+ KB


In [8]:
movies.isnull().sum()
movies = movies.dropna()
movies.isnull().sum()

genres      0
movie_id    0
keywords    0
overview    0
title       0
cast        0
crew        0
dtype: int64

In [9]:
import ast
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L 

In [10]:
movies['genres']= movies['genres'].apply(convert)

In [11]:
movies['keywords'] = movies['keywords'].apply(convert)

In [12]:
# now for cast we want only top 3 name of the actors
def convertcast(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L 

In [13]:
movies['cast'] = movies['cast'].apply(convertcast)

In [14]:
# we want to fetch only the director name out of crew 
def fetch_director_crew(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L 

In [15]:
movies['crew'] = movies['crew'].apply(fetch_director_crew)

In [16]:
movies.head(1)

,genres,movie_id,keywords,overview,title,cast,crew
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...",Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [17]:
# now we have to convert overview which is string to list so that we can concat it to the other columns 
movies['overview'] = movies['overview'].apply(lambda x:x.split()) # this splits each word seperately 

In [18]:
movies.head(1)

,genres,movie_id,keywords,overview,title,cast,crew
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...","[In, the, 22nd, century,, a, paraplegic, Marin...",Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [19]:
#Now we will create a new feature call tags where, overview, genres,keywords cast and crew will be there
movies['tags'] = movies['overview'] + movies['genres']+ movies['keywords']+ movies['cast']+ movies['crew']

In [20]:
movies.head(1)

,genres,movie_id,keywords,overview,title,cast,crew,tags
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...","[In, the, 22nd, century,, a, paraplegic, Marin...",Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."


In [21]:
# Now we do not want the the features before we concat so we can remove that as we have in tags
new_df = movies[['movie_id', 'title', 'tags']]
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [22]:
# Now in tag we have to convert List into string
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

C:\Users\lenovo\AppData\Local\Temp\ipykernel_23688\834465795.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [23]:
from sentence_transformers import SentenceTransformer,util
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
movie_embedding = model.encode(
    new_df['tags'].tolist(),
    show_progress_bar = True
)

Batches:   0%|          | 0/151 [00:00<?, ?it/s]

In [25]:
query = "alien planet"

In [26]:
query_embedding = model.encode(query)

In [27]:
cos_sim = util.cos_sim(query_embedding, movie_embedding)
print("Cosine Similarity:", cos_sim)

Cosine Similarity: tensor([[0.4811, 0.2115, 0.1741,  ..., 0.0369, 0.0702, 0.0215]])


In [28]:
 sorted(list(enumerate(cos_sim[0])), reverse = True, key = lambda x:x[1])[:5]

[(754, tensor(0.4869)),
 (0, tensor(0.4811)),
 (3160, tensor(0.4717)),
 (778, tensor(0.4709)),
 (4703, tensor(0.4700))]

In [29]:
movie_list = sorted(list(enumerate(cos_sim[0])), reverse = True, key = lambda x:x[1])[:5]

In [30]:
for i in movie_list:
    print(new_df.iloc[i[0]].title)

Planet 51
Avatar
Alien
Meet Dave
Another Earth


In [31]:
def recommend_semantic(query):
    query_embedding = model.encode(query)

    cos_sim = util.cos_sim(query_embedding, movie_embedding)

    movie_list = sorted(
        list(enumerate(cos_sim[0])),
        reverse=True,
        key=lambda x: x[1]
    )[:5]

    recommendations = []

    for i in movie_list:
        recommendations.append(new_df.iloc[i[0]].title)

    return recommendations

In [32]:
movies = recommend_semantic("alien planet")
print(movies)

['Planet 51', 'Avatar', 'Alien', 'Meet Dave', 'Another Earth']


In [34]:
import pickle 
pickle.dump(movie_embedding,open('movie_embedding.pkl','wb'))

In [37]:
pickle.dump(new_df, open("movies_semantic.pkl", "wb"))